# Performance Counter Data Conversion

## Convert the Raw Perf Files in to a CSV

This script processes perf-style data files, extracting timestamps and event counts, and outputs a pivoted CSV file with timestamps as rows and events as columns.

## Data Processing Implementation

The core of this notebook is the `process_file` function that:
- Reads raw performance counter data in perf format
- Parses each line to extract timestamps, counts, and event names
- Filters out comments, blank lines, and malformed data
- Organizes the data into a structured dictionary
- Sorts timestamps chronologically and event names alphabetically
- Writes a properly formatted CSV file with consistent structure

## Batch Processing

This section processes all perf files in the RAW_FILES directory:
- Reads each file from the input directory
- Creates corresponding CSV files in the Data directory
- Preserves the base filename while changing the extension
- Provides progress feedback during processing

## Class Labeling for Machine Learning

The final step adds classification labels to each CSV file:
- Examines each filename to determine if it represents clean or malware data
- Adds a "class" column to the CSV header
- Assigns class value 0 to clean samples and 1 to malware samples
- Updates each CSV file with the appropriate classification
- Maintains the same file structure and location

This processed data provides the foundation for the machine learning models that will detect malware based on hardware performance counter patterns.

In [1]:

import os
import csv
import argparse
from collections import defaultdict, OrderedDict

def process_file(in_path, out_path):
    """
    Reads a perf-style data file and writes a pivoted CSV:
      - rows: unique timestamps
      - columns: distinct event names
      - cell value: the count for that event at that timestamp (0 if missing)
    """
    data = defaultdict(dict)  # { timestamp: { event: count, ... }, ... }
    events = set()

    with open(in_path, 'r') as f:
        for line in f:
            line = line.strip()
            # skip blank lines and full-line comments
            if not line or line.startswith('#'):
                continue

            # drop inline comments
            line = line.split('#', 1)[0].strip()
            if not line:
                continue

            parts = line.split()
            # Expect at least: time, counts, event_name
            if len(parts) < 3:
                continue

            time_str = parts[0]
            count_str = parts[1].replace(',', '')
            event_str = parts[2]

            try:
                count = int(count_str)
            except ValueError:
                # skip malformed lines
                continue

            # record
            data[time_str][event_str] = count
            events.add(event_str)

    # sort timestamps (as floats) and events alphabetically
    sorted_times = sorted(data.keys(), key=lambda t: float(t))
    sorted_events = sorted(events)

    # write CSV
    with open(out_path, 'w', newline='') as fout:
        writer = csv.writer(fout)
        # header
        writer.writerow(['time'] + sorted_events)

        for ts in sorted_times:
            row = [ts]
            for ev in sorted_events:
                row.append(data[ts].get(ev, 0))
            writer.writerow(row)



This script processes perf-style data files, extracting timestamps and event counts, and outputs a pivoted CSV file with timestamps as rows and events as columns.    

In [2]:
in_dir = "./RAW_FILES"
out_dir = "./Data"

if not os.path.isdir(in_dir):
    raise NotADirectoryError(f"Input path is not a directory: {in_dir}")
os.makedirs(out_dir, exist_ok=True)

for fname in os.listdir(in_dir):
    in_path = os.path.join(in_dir, fname)
    if not os.path.isfile(in_path):
        continue

    base, _ = os.path.splitext(fname)
    out_fname = base + '.csv'
    out_path = os.path.join(out_dir, out_fname)

    print(f"Processing {fname} → {out_fname}")
    process_file(in_path, out_path)


Processing clean.txt → clean.csv
Processing malware.txt → malware.csv


In [5]:
# To each of the CSV files, we apeend the following header: "class"
# each row in this column will have 0 for the 'clean' csv, 1 for the 'malware' csv

for fname in os.listdir(out_dir):
    in_path = os.path.join(out_dir, fname)
    if not os.path.isfile(in_path):
        continue

    # Determine class based on filename
    if 'clean' in fname.lower():
        class_value = 0
    elif 'malware' in fname.lower():
        class_value = 1
    else:
        continue  # Skip files that don't match expected naming

    # Read existing CSV and write with new class column
    with open(in_path, 'r') as f:
        reader = csv.reader(f)
        rows = list(reader)

    # Add class column header
    rows[0].append('class')

    # Add class value to each row
    for row in rows[1:]:
        row.append(class_value)

    # Write back to the same file
    with open(in_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerows(rows)